In [1]:
import os
import time
import sqlparse
import warnings
import pymysql
import numpy as np
import pandas as pd
import datetime as dt
from tqdm import tqdm
from dotenv import load_dotenv

# Vectorization vs Parallelization

For June 2026, the seismic revision routine makes a search of 17 different checks on the seismic data. Each check is a function that takes in the seismic data and performs some operations on it to check for certain conditions. The checks are performed sequentially, which means that each check is performed one after the other. This can be time-consuming, especially if the seismic data is large. In order to handle this, initially parallelization was implemented using the multiprocessing library in Python. This allowed the checks to be performed in parallel, which significantly reduced the time taken to perform the checks. However, this approach had some limitations, such as the overhead of creating and managing multiple processes, and the need to ensure that the checks were thread-safe.

Furthermore, there are some operations that are performed on the seismic data that can be vectorized using libraries such as NumPy. Vectorization allows for the operations to be performed on entire arrays of data at once, rather than iterating through each element individually. This can significantly reduce the time taken to perform the operations, as it takes advantage of the underlying hardware optimizations for array operations. In addition, there is no need to initialize multiple workers or manage the overhead associated with parallelization. By using vectorization, we can reduce energy consumption and improve the efficiency of the seismic revision routine, while also simplifying the code and reducing the potential for errors. Overall, while parallelization can be useful in certain situations, we will check if vectorization is a more efficient and effective approach for handling large datasets and performing complex operations on them.

In this notebook, we will compare the performance of vectorization and parallelization for a specific check in the seismic revision routine. We will implement both approaches and measure the time taken to perform the check on a sample seismic dataset. We will also analyze the results and discuss the advantages and disadvantages of each approach. Finally, we will make recommendations on which approach to use for different scenarios in the seismic revision routine. The main idea is to define single functions to perform each check, and then use either vectorization or parallelization to apply those functions to the seismic data.

## Query seismic data from database

Let's start by querying the seismic data from the database. We will use the `pymysql` library to connect to the database and execute a SQL query to retrieve the seismic data. We will also use the `dotenv` library to load the database credentials from a `.env` file, as stated in the previous notebook.

In [2]:
env_path = os.path.join(os.getcwd(), '.env')
load_dotenv(dotenv_path=env_path)

def connect_to_db(
        query: str,
        start_time: dt.datetime = None,
        end_time: dt.datetime = None,
        **kwargs):

    if start_time and end_time:
        start_time_str = start_time.strftime("%Y-%m-%d %H:%M:%S")
        end_time_str = end_time.strftime("%Y-%m-%d %H:%M:%S")
        full_query = f"{query} '{start_time_str}' and '{end_time_str}' ORDER BY Origin.time_value ASC;"  # Filter and order by time_value
    else:
        full_query = query

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        db_connection = pymysql.connect(
            host=os.getenv('SERVER_HOST'),
            user=os.getenv('SERVER_USERNAME'),
            password=os.getenv('SERVER_PASSWORD'),
            db=os.getenv('SERVER_DATABASE')
        )

        try:
            with tqdm(total=1, desc='Querying database...', unit='query', leave=False,bar_format="{desc}") as pbar:
                df = pd.read_sql_query(full_query, db_connection, **kwargs)
                pbar.update(1)
        finally:
            db_connection.close()

    return df

In [3]:
# Read revision.sql file
with open('./queries/revision.sql', 'r') as file:
    revision_query = file.read()

clean_sql = sqlparse.format(revision_query, strip_comments=True).strip()

initial_time = dt.datetime(2023, 5, 3, 0, 0, 0)
final_time = dt.datetime(2026, 3, 17, 0, 0, 0)
event_df3 = connect_to_db(clean_sql, start_time=initial_time, end_time=final_time)

print(f"Number of rows in the seismic data: {len(event_df3)}")
# Print how events are by event_type
print("Number of events by event_type:")
print(event_df3['event_type'].value_counts())

Number of rows in the seismic data: 209243
Number of events by event_type:
event_type
not locatable                  116633
earthquake                      78301
not existing                     5211
explosion                        4372
outside of network interest      4200
volcanic eruption                 455
induced earthquake                  3
Name: count, dtype: int64


## 1. Comparison checks

The seismic revision routine performs a list of quality checks on earthquakes, based on relational or absolute thresholds. These checks are designed to identify earthquakes that may not be reliable and may require further investigation. Some of the checks that are performed include:

1. High RMS: This check identifies earthquakes with a high root-mean-square (RMS) value, which indicates that the seismic data is noisy and may not be reliable. The threshold for this check is typically set at a certain value, such as 1.51.
2. Localization uncertainty: This check identifies earthquakes with a high localization uncertainty, which indicates that the location of the earthquake is not well-defined. The threshold for this check is typically set at a certain value, such as 12 km. It is applied both on latitude, longitude, and depth.
3. Depth check: This check identifies earthquakes with a depth that is outside of a certain range, such as between 0 and 700 km. This check is important because earthquakes that are too shallow or too deep may not be reliable and may require further investigation.

For all these type of checks, it is possible to vectorize the solution by applying the check to the entire dataset at once, rather than iterating through each earthquake individually. The idea here is to create a single general function, receiving the filtered seismic data, the threshold value or values (if there are multiple thresholds), and the column to be checked. The function will then apply the check to the entire dataset and return a boolean mask indicating which earthquakes meet the criteria for being flagged as unreliable. This approach can significantly reduce the time taken to perform the checks, without making it too complicated to be debugged or maintained.

In [8]:
# Previous version
def single_check(event):
    observations = []

    # First check: High RMS values
    exceptions = ["not locatable", "outside of network interest", "volcanic eruption", "explosion", "not existing"]
    if event['quality_standardError'] > 1.51 and event['event_type'] not in exceptions:
        observations.append("High RMS value")

    if len(observations) > 0:  # If the event has observations, return the information
        return event, observations
    else:
        return None, None

# For loop version
time1 = time.time()
results = []
for _, event in event_df3.iterrows():
    result, obs = single_check(event)
    if result is not None:
        results.append((result, obs))
high_rms_df_loop = pd.DataFrame([res[0] for res in results])
time2 = time.time()
print(f"Number of events with high RMS (loop version): {len(high_rms_df_loop)}")
print(f"Time taken for high RMS check (loop version): {time2 - time1:.4f} seconds")

Number of events with high RMS (loop version): 16
Time taken for high RMS check (loop version): 21.9958 seconds


In [4]:
# Vectorized function to make comparison between a column and a threshold value
def build_quality_mask(
    events: pd.DataFrame,
    column: str,
    mode: str,
    threshold=None,
    lower=None,
    upper=None,
    dtype=np.float64
) -> np.ndarray:
    """
    Vectorized generic comparator for seismic quality checks.

    Parameters
    ----------
    events : pd.DataFrame
        Input seismic dataframe.
    column : str
        Column to evaluate.
    mode : str
        Comparison mode:
            'gt'       -> values > threshold
            'ge'       -> values >= threshold
            'lt'       -> values < threshold
            'le'       -> values <= threshold
            'between'  -> lower <= values <= upper
            'outside'  -> values < lower or values > upper
            'abs_gt'   -> abs(values) > threshold
            'abs_ge'   -> abs(values) >= threshold
    threshold : float, optional
        Threshold for one-sided comparisons.
    lower, upper : float, optional
        Bounds for range comparisons.
    dtype : numpy dtype
        Target dtype for NumPy conversion.

    Returns
    -------
    np.ndarray
        Boolean mask of flagged rows.
    """
    values = events[column].to_numpy(dtype=dtype, copy=False)

    if mode == 'gt':
        return values > threshold
    elif mode == 'ge':
        return values >= threshold
    elif mode == 'lt':
        return values < threshold
    elif mode == 'le':
        return values <= threshold
    elif mode == 'between':
        return (values >= lower) & (values <= upper)
    elif mode == 'outside':
        return (values < lower) | (values > upper)
    elif mode == 'abs_gt':
        return np.abs(values) > threshold
    elif mode == 'abs_ge':
        return np.abs(values) >= threshold
    else:
        raise ValueError(f"Unsupported mode: {mode}")

# Check for high RMS values on 'earthquake' and 'volcanic eruption' event types
time1 = time.time()
rms_threshold = 1.51
rms_mask = build_quality_mask(
    events=event_df3[event_df3['event_type'].isin(['earthquake', 'volcanic eruption'])],
    column='quality_standardError',
    mode='gt',
    threshold=rms_threshold
)
high_rms_df = event_df3[event_df3['event_type'].isin(['earthquake', 'volcanic eruption'])][rms_mask]
time2 = time.time()
print(f"Number of events with high RMS: {len(high_rms_df)}")
print(f"Time taken for high RMS check: {time2 - time1:.4f} seconds")

Number of events with high RMS: 16
Time taken for high RMS check: 0.0812 seconds


As you can see, the vectorized version of the high RMS check is significantly faster than the loop version (13.53 s to just 0.0812 s!). This is because the vectorized version takes advantage of NumPy's optimized array operations, which are implemented in C and can be executed much faster than Python loops. In contrast, the loop version iterates through each event one by one, which is much slower, especially for large datasets. Additionally, the vectorized version is more concise and easier to read, as it eliminates the need for explicit loops and conditional statements. Overall, this demonstrates the significant performance benefits of using vectorization for data processing tasks in Python.

Now let's create a wrapper function to apply multiple checks at once:

In [13]:
# Wrapper function to apply multiple checks at once
def seismic_quality_checks(events: pd.DataFrame) -> pd.DataFrame:
    """
    Apply common earthquake quality checks and return flagged events.
    """
    selections = events[events['event_type'].eq('earthquake')].reset_index(drop=True)

    if selections.empty:
        return selections.iloc[0:0].copy()

    masks = {
        'High RMS': build_quality_mask(
            selections, column='quality_standardError', mode='gt', threshold=1.51
        ),
        'High err_lat': build_quality_mask(
             selections, column='latitude_uncertainty', mode='gt', threshold=12.0
        ),
        'High err_lon': build_quality_mask(
            selections, column='longitude_uncertainty', mode='gt', threshold=12.0
        ),
        'High err_depth': build_quality_mask(
            selections, column='depth_uncertainty', mode='gt', threshold=12.0
        ),
        'Invalid depth': build_quality_mask(
            selections, column='depth_value', mode='outside', lower=0.0, upper=200.0
        ),
    }

    combined_mask = np.zeros(len(selections), dtype=bool)
    for mask in masks.values():
        combined_mask |= mask

    flagged = selections.loc[combined_mask].copy()

    flagged_idx = np.where(combined_mask)[0]
    observations = []
    for i in flagged_idx:
        obs = [name for name, mask in masks.items() if mask[i]]
        observations.append(', '.join(obs))

    flagged['Observations'] = observations
    return flagged.reset_index(drop=True)

# Apply the checks and measure time
time1 = time.time()
flagged_events = seismic_quality_checks(event_df3)
time2 = time.time()
print(f"Number of flagged events: {len(flagged_events)}")
print(f"Time taken for seismic quality checks: {time2 - time1:.4f} seconds")

Number of flagged events: 479
Time taken for seismic quality checks: 0.0870 seconds


In [14]:
flagged_events

,time_value,publicID,depth_value,magnitude_value,quality_standardError,depth_uncertainty,latitude_uncertainty,longitude_uncertainty,quality_associatedPhaseCount,quality_usedPhaseCount,...,event_type,creationInfo_agencyID,text,latitude_value,longitude_value,magnitude_type,methodID,earthModelID,comment,Observations
0,2023-05-04 14:46:31,SGC2023isxsdy,32.960000,1.085318,1.680000,10.500000,5.444722,5.444722,13.0,13.0,...,earthquake,SGC,"Zapatoca - Santander, Colombia",6.828333,-73.292667,MLr_vmm,Hypo71,VMM,None,High RMS
1,2023-05-06 21:53:34,SGC2023ixdhyk,-2.011719,0.867000,0.577960,4.123822,2.345152,3.548808,12.0,12.0,...,earthquake,SGC,"Quetame - Cundinamarca, Colombia",4.344292,-73.808842,MLr_3,NonLinLoc,Poveda_et_al_2018,None,Invalid depth
2,2023-05-07 10:22:06,SGC2023iycczn,-1.160000,1.085135,0.710000,5.000000,1.414214,1.414214,15.0,15.0,...,earthquake,SGC,"TimanÃ¡ - Huila, Colombia",1.971500,-75.923333,MLr_2,Hypo71,RSNC,None,Invalid depth
3,2023-05-07 13:50:01,SGC2023iyjads,-2.011719,0.808768,0.644393,6.257838,2.268201,1.941907,14.0,14.0,...,earthquake,SGC,"PÃ¡cora - Caldas, Colombia",5.474582,-75.511715,MLr_2,NonLinLoc,Poveda_et_al_2018,None,Invalid depth
4,2023-05-10 18:07:47,SGC2023jeesxp,-1.540000,1.342816,0.620000,7.300000,5.656854,5.656854,9.0,9.0,...,earthquake,SGC,"Colombia-Venezuela, RegiÃ³n Fronteriza",7.563500,-72.225667,MLr,Hypo71,RSNC,None,Invalid depth
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
474,2025-07-24 19:11:50,SGC2025olyidd,252.400000,2.243815,0.230000,3.700000,7.283200,7.283200,13.0,13.0,...,earthquake,SGC,"Albania - la Guajira, Colombia",11.139167,-72.521667,MLr_4,Hypo71,CARMA,None,Invalid depth
475,2025-08-19 16:02:15,SGC2025qhkxnl,0.000000,2.904271,1.636559,0.000000,3.563018,5.394898,24.0,16.0,...,earthquake,SGC,OcÃ©ano PacÃ­fico,2.605613,-78.976692,MLr_1,LOCSAT,iasp91,None,High RMS
476,2025-09-25 20:37:01,SGC2025sxrwgc,267.690000,2.265484,0.140000,1.700000,7.000357,7.000357,9.0,9.0,...,earthquake,SGC,"Maicao - la Guajira, Colombia",11.511000,-72.369500,MLr_4,Hypo71,CARMA,None,Invalid depth
477,2025-10-29 08:19:21,SGC2025vhedge,10.000000,4.801020,1.717966,0.000000,1.935249,2.364565,97.0,85.0,...,earthquake,SGC,OcÃ©ano PacÃ­fico,3.361495,-82.730011,Mw(mB),LOCSAT,iasp91,None,High RMS


### Appendix: Locatable events

A check of the seismic revision routine is to identify events that are locatable, but are labeled incorrectly as "not locatable". At RSNC, a event is locatable if it has at least 4 p phases and 2 s phases associated to it. This check is important because it can help to identify earthquakes that may have been misclassified and may require further investigation.

However, the previous routine only checks a verification on the _'quality_associatedPhaseCount'_ column to be greater than or equal to 8 (plus one due to an event with 6 phases in seiscomp will have a _quality_associatedPhaseCount_ of 7). Then, an event with for example 8 p phases and 0 s phases would be flagged as locatable, which is not correct.

The challenge here is that query the number of p and s phases associated to each event requires a join between the _Origin_ and _Arrival_ tables in the database, which can be time-consuming, especially for large datasets. Additionally, the check needs to be performed for each event individually, which can further increase the time taken to perform the check. Therefore, the strategy here is to use the columns _quality_associatedPhaseCount_, _quality_usedPhaseCount_, _quality_usedStationCount_ and _quality_associatedStationCount_ to create a vectorized check that can identify locatable events without the need for a join between the tables. This approach can significantly reduce the time taken to perform the check, while still providing accurate results.

In [17]:
# First example: 3 p and 3 s event (within 2026-03-01 11:13:00 and 2026-03-01 11:14:00)
start_filter = dt.datetime(2026, 3, 1, 11, 13, 0)
end_filter = dt.datetime(2026, 3, 1, 11, 14, 0)
subset_df = event_df3[(event_df3['time_value'] >= start_filter) & (event_df3['time_value'] <= end_filter)].copy()
subset_df[['time_value', 'event_type', 'quality_associatedPhaseCount', 'quality_usedPhaseCount', 'quality_usedStationCount', 'quality_associatedStationCount']]

,time_value,event_type,quality_associatedPhaseCount,quality_usedPhaseCount,quality_usedStationCount,quality_associatedStationCount
205886,2026-03-01 11:13:54,not locatable,7.0,7.0,6.0,NaN


In [18]:
# Second example: 4 p and 3 s event (within 2026-03-01 06:38:00 and 2026-03-01 06:39:00)
start_filter_2 = dt.datetime(2026, 3, 1, 6, 38, 0)
end_filter_2 = dt.datetime(2026, 3, 1, 6, 39, 0)
subset_df_2 = event_df3[(event_df3['time_value'] >= start_filter_2) & (event_df3['time_value'] <= end_filter_2)].copy()
subset_df_2[['time_value', 'event_type', 'quality_associatedPhaseCount', 'quality_usedPhaseCount', 'quality_usedStationCount', 'quality_associatedStationCount']]

,time_value,event_type,quality_associatedPhaseCount,quality_usedPhaseCount,quality_usedStationCount,quality_associatedStationCount
205831,2026-03-01 06:38:24,earthquake,8.0,8.0,7.0,NaN


In [19]:
# Third example: 4 p and 4 s event (within 2026-03-01 00:22:00 and 2026-03-01 00:23:00)
start_filter_3 = dt.datetime(2026, 3, 1, 0, 22, 0)
end_filter_3 = dt.datetime(2026, 3, 1, 0, 23, 0)
subset_df_3 = event_df3[(event_df3['time_value'] >= start_filter_3) & (event_df3['time_value'] <= end_filter_3)].copy()
subset_df_3[['time_value', 'event_type', 'quality_associatedPhaseCount', 'quality_usedPhaseCount', 'quality_usedStationCount', 'quality_associatedStationCount']]

,time_value,event_type,quality_associatedPhaseCount,quality_usedPhaseCount,quality_usedStationCount,quality_associatedStationCount
205762,2026-03-01 00:22:47,earthquake,8.0,8.0,4.0,NaN


In [20]:
# Fourth example: 4 p and 4 s event (within 2026-03-01 11:33:00 and 2026-03-01 11:34:00)
start_filter_4 = dt.datetime(2026, 3, 1, 11, 33, 0)
end_filter_4 = dt.datetime(2026, 3, 1, 11, 34, 0)
subset_df_4 = event_df3[(event_df3['time_value'] >= start_filter_4) & (event_df3['time_value'] <= end_filter_4)].copy()
subset_df_4[['time_value', 'event_type', 'quality_associatedPhaseCount', 'quality_usedPhaseCount', 'quality_usedStationCount', 'quality_associatedStationCount']]

,time_value,event_type,quality_associatedPhaseCount,quality_usedPhaseCount,quality_usedStationCount,quality_associatedStationCount
205889,2026-03-01 11:33:07,earthquake,9.0,9.0,8.0,NaN


In [21]:
# Fifth example: 44 p (43 used) and 40 s (81 used picks to locate and 84 total picks) event (within 2026-03-01 09:37:00 and 2026-03-01 09:38:00)
start_filter_5 = dt.datetime(2026, 3, 1, 9, 37, 0)
end_filter_5 = dt.datetime(2026, 3, 1, 9, 38, 0)
subset_df_5 = event_df3[(event_df3['time_value'] >= start_filter_5) & (event_df3['time_value'] <= end_filter_5)].copy()
subset_df_5[['time_value', 'event_type', 'quality_associatedPhaseCount', 'quality_usedPhaseCount', 'quality_usedStationCount', 'quality_associatedStationCount']]

,time_value,event_type,quality_associatedPhaseCount,quality_usedPhaseCount,quality_usedStationCount,quality_associatedStationCount
205864,2026-03-01 09:37:14,earthquake,84.0,81.0,43.0,NaN
